# 🎯 Module 11: Tool Calling and External APIs

> **Course:** Generative AI — From Scratch to GenAI Engineer  
> **Stage:** Stage 3 — LLM Application Engineering  
> **Module:** 11 — Tool Calling and External APIs  
> **Planned Date:** August 5, 2026  
> **Level:** Intermediate → Advanced  
> **Prerequisites:** LLM APIs, Prompt Engineering, Structured Outputs, LangChain basics  
> **Previous:** 10 — LLM Frameworks and LangChain  
> **Next:** 12 — LLM Evaluation and Guardrails

---

Now we connect models to the **real world** — safely.

**Tool calling** lets an LLM *request* an operation. **Your application decides** whether to run it.

In this module we'll cover:

- What **tool calling** is
- The **lifecycle** (request → validate → execute → return → respond)
- **Tool calling vs structured outputs vs RAG**
- Defining **tools** and **schemas**
- **Argument validation** with Pydantic
- A working **tool-calling example** with the OpenAI SDK
- A **controlled tool-calling loop**
- Connecting to **external REST APIs**
- **Security** — allowlists, authorization, SSRF, side-effects
- **Error handling** and **testing**
- A **final project**: API-Connected AI Assistant

## 📚 Learning Objectives

By the end of this module you should be able to:

1. Explain **function calling** and **tool calling**
2. Distinguish *generating text* from *requesting a tool*
3. Define tools with **names, descriptions, and argument schemas**
4. Understand the **tool-calling lifecycle**
5. Execute tool calls via an explicit **allowlist**
6. Connect an LLM with **external REST APIs**
7. **Validate arguments** and **handle failures**
8. Understand **sequential** and **multiple** tool calls
9. Build a **controlled tool-calling loop**
10. Apply **authorization, input validation, approval requirements**
11. Understand how tools fit into larger AI applications
12. Prepare for **evaluation, guardrails, and agentic workflows**

---
## ⚙️ 0. Setup & Imports

This notebook is **safe to run without an API key** — every tool-calling step is shown, and the model calls are guarded.

In [22]:
import os
import json
import time
from pathlib import Path
from typing import Any

HAS_KEY   = bool(os.getenv('OPENAI_API_KEY'))
HAS_MODEL = bool(os.getenv('OPENAI_MODEL')) or bool(os.getenv('LLM_MODEL'))

print('✅ Setup ready.')
print(f'OPENAI_API_KEY present: {HAS_KEY}')
print(f'OPENAI_MODEL present  : {HAS_MODEL}')

try:
    import pydantic
    print(f'Pydantic version: {pydantic.__version__}')
    HAS_PYDANTIC = True
except ImportError:
    print('Pydantic not installed. Run: pip install pydantic')
    HAS_PYDANTIC = False

✅ Setup ready.
OPENAI_API_KEY present: False
OPENAI_MODEL present  : False
Pydantic not installed. Run: pip install pydantic


---
## 🧠 1. What Is Tool Calling?

**Tool calling** lets a language model **request** that an application run a function or external operation.

A tool might:

- Calculate a value
- Search a knowledge base
- Retrieve an order
- Check inventory
- Query a database
- Call a weather / currency API
- Create a ticket
- Send a notification (after authorization)

> 🔑 **Key distinction:**  
> **The model requests a tool call. Your application executes it.**

The model does **not** automatically gain access to your Python env, DB, filesystem, or external services.

### Example

User: *"What is 125 multiplied by 8?"*

The model may request:

```json
{
  "name": "multiply",
  "arguments": { "a": 125, "b": 8 }
}
```

**Your app** validates and runs:

```python
def multiply(a, b): return a * b
multiply(125, 8)   # → 1000
```

Then sends `1000` back to the model so it can formulate the final response.

---
## 🔄 2. The Tool-Calling Lifecycle

```text
1. User Sends a Question
           │
           ▼
2. Application Sends Messages + Tool Definitions
           │
           ▼
3. Model Requests a Tool Call
           │
           ▼
4. Application Validates the Request
           │
           ▼
5. Application Executes the Approved Function
           │
           ▼
6. Application Sends the Tool Result Back
           │
           ▼
7. Model Generates the Final Answer
```

### Important details

1. A **tool definition** describes what the tool does and what args it accepts.
2. The model **can request** a tool — but the request is **not proof** the args are correct.
3. **Your app** decides whether the requested tool is allowed.
4. The function runs in **your app's execution environment**.
5. The result is returned using the **provider's required tool-result format**.
6. The model can use that result or request another tool **if the app allows it**.

> 🎯 Tool calling is an **application-control feature**, not just a prompt that says *"use this function"*.

---
## ⚖️ 3. Tool Calling vs Structured Outputs vs RAG

| Concept | Main purpose | Example |
| --- | --- | --- |
| **Structured outputs** | Return data in a defined schema | Extract a support ticket as JSON |
| **Tool calling** | Request an operation / function | Look up an order |
| **RAG** | Retrieve info from a knowledge source | Answer from company docs |
| **Agentic workflow** | Coordinate steps toward a goal | Research using multiple tools |

- A **structured response** *describes information*.
- A **tool call** *requests an operation*.
- RAG **can** use tool calling — but doesn't have to.
- A tool-calling app can be a simple, deterministic loop **without** being an autonomous agent.

---
## 🛠️ 4. Define a Simple Tool

Start with **ordinary Python functions**.

```python
def get_order_status(order_id: str) -> dict:
    """Return the status of a known order."""
    orders = {
        'ORD-1001': {'status': 'shipped',    'eta': '2026-08-07'},
        'ORD-1002': {'status': 'processing', 'eta': None},
    }
    order = orders.get(order_id)
    if order is None:
        return {'found': False, 'message': 'Order not found.'}
    return {'found': True, 'order_id': order_id, **order}
```

### What makes a good tool

- Clear **name**
- **Single responsibility**
- Concise **description**
- **Explicit arguments**
- **Predictable output**
- **Defined error behavior**
- Appropriate **access controls**

In [23]:
# Tool implementation (works without any API key)
def get_order_status(order_id: str) -> dict:
    """Return the status of a known order."""
    orders = {
        'ORD-1001': {'status': 'shipped',    'estimated_delivery': '2026-08-07'},
        'ORD-1002': {'status': 'processing', 'estimated_delivery': None},
    }
    order = orders.get(order_id)
    if order is None:
        return {'found': False, 'message': 'Order not found.'}
    return {'found': True, 'order_id': order_id, **order}


print(get_order_status('ORD-1001'))
print(get_order_status('ORD-9999'))

{'found': True, 'order_id': 'ORD-1001', 'status': 'shipped', 'estimated_delivery': '2026-08-07'}
{'found': False, 'message': 'Order not found.'}


---
## 📋 5. Tool Schemas & Argument Validation

The model needs to know **which arguments** the tool accepts.

```json
{
  "name": "get_order_status",
  "description": "Look up the status of an order.",
  "parameters": {
    "type": "object",
    "properties": {
      "order_id": {
        "type": "string",
        "description": "The order identifier."
      }
    },
    "required": ["order_id"],
    "additionalProperties": false
  }
}
```

> ⚠️ Schemas help but do **not** replace server-side validation.

Your app must still check:

- Required fields
- Argument types
- Allowed values
- String lengths / numeric ranges
- Resource ownership & permissions
- Whether the operation is appropriate for the user

In [24]:
# Validate arguments with Pydantic
if HAS_PYDANTIC:
    from pydantic import BaseModel, Field, ValidationError

    class OrderLookupInput(BaseModel):
        order_id: str = Field(
            min_length=1,
            max_length=40,
            pattern=r'^ORD-[0-9]+$',
        )

    # Valid case
    v = OrderLookupInput(order_id='ORD-1001')
    print('✅ Valid:', v.model_dump())

    # Invalid cases
    for bad in [
        {'order_id': '1001'},        # missing ORD- prefix
        {'order_id': 'ORD-abc'},     # not digits
        {'order_id': ''},            # empty
    ]:
        try:
            OrderLookupInput(**bad)
            print(f'⚠️  Should have failed: {bad}')
        except ValidationError as e:
            print(f"✅ Correctly rejected {bad}: {e.errors()[0]['type']}")
else:
    print('Skipped — install pydantic.')

Skipped — install pydantic.


---
## 💻 6. First Practical — Tool Calling With OpenAI

We use the **OpenAI Python SDK directly** so you understand the protocol **before** adding framework abstractions.

### Install
```bash
pip install openai python-dotenv
```

### `.env`
```dotenv
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=your_supported_model_name
```

### Flow

1. Send messages + tool definitions
2. Receive a **tool_call** request
3. **Check the tool name** against an allowlist
4. **Parse** arguments (JSON)
5. **Validate** arguments
6. **Execute** the function
7. Append the **tool result** with its matching `tool_call_id`
8. Send messages **again** for the final answer

In [25]:
# Full tool-calling example — runs only if credentials exist
def run_tool_calling_demo():
    if not (HAS_KEY and HAS_MODEL):
        print('Skipped — configure OPENAI_API_KEY and OPENAI_MODEL to run this live.')
        return
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
        load_dotenv()

        client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        MODEL = os.environ['OPENAI_MODEL']

        def multiply(a: float, b: float) -> float:
            return a * b

        tools = [{
            'type': 'function',
            'function': {
                'name': 'multiply',
                'description': 'Multiply two numbers.',
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'a': {'type': 'number', 'description': 'First number.'},
                        'b': {'type': 'number', 'description': 'Second number.'},
                    },
                    'required': ['a', 'b'],
                    'additionalProperties': False,
                },
                'strict': True,
            },
        }]

        ALLOWED = {'multiply': multiply}

        messages = [{'role': 'user', 'content': 'What is 125 multiplied by 8?'}]
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, tool_choice='auto',
        )
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        if assistant_message.tool_calls:
            for tc in assistant_message.tool_calls:
                if tc.function.name not in ALLOWED:
                    raise ValueError(f'Tool not allowed: {tc.function.name}')
                args = json.loads(tc.function.arguments)
                result = ALLOWED[tc.function.name](**args)
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tc.id,
                    'content': json.dumps({'result': result}),
                })

            final = client.chat.completions.create(model=MODEL, messages=messages)
            print('Final answer:', final.choices[0].message.content)
        else:
            print('No tool call — model answered directly:')
            print(assistant_message.content)
    except Exception as e:
        print(f'Live tool-calling demo failed: {type(e).__name__}: {e}')

run_tool_calling_demo()

Skipped — configure OPENAI_API_KEY and OPENAI_MODEL to run this live.


---
## 🔁 7. Build a Controlled Tool-Calling Loop

Real apps may need **more than one** tool call.

```text
Send Messages + Tools
        │
        ▼
Receive Model Response
        │
        ▼
Any Tool Calls?
   ┌────┴─────┐
   │          │
  No         Yes
   │          │
   ▼          ▼
Return     Validate Each Call
Answer          │
                ▼
          Check Allowlist
                │
                ▼
          Execute Tool
                │
                ▼
          Append Results
                │
                └──────► Send Messages Again
```

### Loop design requirements

1. **Max iterations** (prevent infinite loops)
2. Only **registered tools**
3. **Validate every arg**
4. Match each result to the right **`tool_call_id`**
5. Handle tool failures without crashing
6. Prevent duplicate **side-effecting** operations
7. Return a controlled error on max-iteration

> ⚠️ Max iterations prevents a runaway loop — but it is **not** a security control.

In [26]:
# Registry pattern — the app's authoritative tool allowlist
TOOL_REGISTRY = {}

def register_tool(name):
    def deco(fn):
        TOOL_REGISTRY[name] = fn
        return fn
    return deco

@register_tool('multiply')
def multiply(a: float, b: float) -> float:
    return a * b

@register_tool('get_order_status')
def get_order_status_tool(order_id: str) -> dict:
    return get_order_status(order_id)

print('Registered tools:', list(TOOL_REGISTRY.keys()))

def execute_tool_call(name, arguments):
    """Execute a validated tool call from the registry."""
    if name not in TOOL_REGISTRY:
        return {'success': False, 'error': f'Unknown tool: {name}'}
    try:
        result = TOOL_REGISTRY[name](**arguments)
        return {'success': True, 'result': result}
    except Exception as e:
        return {'success': False, 'error': f'{type(e).__name__}: {e}'}

print()
print(execute_tool_call('multiply', {'a': 125, 'b': 8}))
print(execute_tool_call('get_order_status', {'order_id': 'ORD-1001'}))
print(execute_tool_call('delete_everything', {}))   # not allowed

Registered tools: ['multiply', 'get_order_status']

{'success': True, 'result': 1000}
{'success': True, 'result': {'found': True, 'order_id': 'ORD-1001', 'status': 'shipped', 'estimated_delivery': '2026-08-07'}}
{'success': False, 'error': 'Unknown tool: delete_everything'}


---
## 🌐 8. Connect Tools to External APIs

A tool can call a **REST API** instead of a local function.

### ⚠️ DO NOT expose arbitrary URLs as a tool

An untrusted model-generated URL can cause a **Server-Side Request Forgery (SSRF)** attack — reaching internal services.

### ✅ Safer pattern

Fix the **service endpoint** in your app; the model provides only **validated parameters**.

```python
import requests

API_BASE_URL = 'https://api.example.com'

def get_product(product_id: str) -> dict:
    if not product_id.isalnum():
        raise ValueError('Invalid product identifier.')
    response = requests.get(
        f'{API_BASE_URL}/products/{product_id}',
        timeout=10,
    )
    response.raise_for_status()
    return response.json()
```

### Best practices

- Use **HTTPS**
- Set **timeouts**
- Handle **non-2xx**
- Validate & limit **user params**
- Store creds in **env / secret manager**
- Restrict **outbound** network
- Avoid returning **unnecessary sensitive** data
- Apply **rate limits**
- Log tool name, outcome, latency
- **Idempotency** for write ops

In [27]:
# Simulated external API call (no network) — demonstrates the tool pattern
import time

def mock_api_get_product(product_id: str) -> dict:
    """Mimics an HTTPS GET to a fixed endpoint."""
    if not product_id.isalnum():
        raise ValueError('Invalid product identifier.')
    time.sleep(0.05)   # simulate network latency
    return {'product_id': product_id, 'name': 'Widget', 'price': 9.99}

print(mock_api_get_product('A123'))

try:
    mock_api_get_product('../../etc/passwd')
except ValueError as e:
    print(f'✅ Correctly rejected malicious input: {e}')

{'product_id': 'A123', 'name': 'Widget', 'price': 9.99}
✅ Correctly rejected malicious input: Invalid product identifier.


---
## 🔐 9. Tool Security & Permissions

Treat **every** requested tool call as **untrusted input**.

### Read vs Write operations

| Tool type | Example | Typical safeguard |
| --- | --- | --- |
| Read-only | Search public docs | Validate input, control data access |
| User-specific read | Get an order | **Auth + authz** |
| Reversible write | Create draft ticket | Validate fields, enforce perms |
| Sensitive write | Issue refund | **Strong auth + approval** |
| Destructive | Delete account | **Explicit confirmation + strict controls** |

### 7 essential security rules

1. **Never execute arbitrary model-generated code** — use a predefined registry.
2. **Never trust the model to authorize itself** — enforce perms in your app.
3. **Keep credentials out of prompts** — secrets belong in secure config.
4. **Validate every argument** — well-formed ≠ authorized.
5. **Separate proposal from execution** — for consequential ops, require confirmation.
6. **Treat tool results as untrusted data** — services can return malicious content.
7. **Preserve auditability** — who, which tool, succeeded, which approval applied.

---
## 🧪 10. Error Handling & Testing

### Failures to expect

- Invalid arguments
- Unknown tool names
- Missing permissions
- Network timeouts
- Rate limits
- External service errors
- Unexpected response formats
- Business-rule violations

### Example — validate before executing

```python
def execute_order_lookup(raw_arguments):
    try:
        validated = OrderLookupInput.model_validate(raw_arguments)
    except ValidationError:
        return {'success': False, 'error': 'Invalid arguments.'}
    result = get_order_status(validated.order_id)
    return {'success': True, 'data': result}
```

> ⚠️ Real apps **also** check authorization before returning an order's data.

### Testing checklist

- ✅ Valid & invalid arguments
- ✅ Unknown tool names
- ✅ Missing required fields
- ✅ Unauthorized access
- ✅ External API timeouts
- ✅ Rate-limit responses
- ✅ Malformed API responses
- ✅ Max loop iterations
- ✅ Duplicate write ops
- ✅ **Mock** external services in unit tests

In [29]:
# A robust tool executor with validation + error handling
if HAS_PYDANTIC:
    from pydantic import BaseModel, Field, ValidationError
    from typing import Literal

    class OrderLookupInput(BaseModel):
        order_id: str = Field(min_length=1, max_length=40, pattern=r'^ORD-[0-9]+$')

    def execute_order_lookup(raw_arguments: dict) -> dict:
        try:
            validated = OrderLookupInput.model_validate(raw_arguments)
        except ValidationError:
            return {'success': False, 'error': 'Invalid order lookup arguments.'}
        result = get_order_status(validated.order_id)
        return {'success': True, 'data': result}

    # Test cases
    for args in [
        {'order_id': 'ORD-1001'},   # valid
        {'order_id': '1001'},       # invalid format
        {},                          # missing
    ]:
        print(f'{args}  →  {execute_order_lookup(args)}')

---
## 🏋️ 11. Final Project — API-Connected AI Assistant

### 🎯 Goal
Build an assistant that answers questions by calling a **small set of controlled tools**.

### Suggested tools

1. `calculate_total` — perform a calculation
2. `search_knowledge_base` — retrieve approved docs
3. `get_order_status` — look up a mock order
4. `create_support_ticket` — create a ticket in a mock service

### Core requirements

- [ ] Define tools with **clear descriptions**
- [ ] Define **argument schemas**
- [ ] Register tools in an **explicit allowlist**
- [ ] **Validate** every model-generated argument
- [ ] **Execute** tools in application code
- [ ] Return tool results using the **correct API protocol**
- [ ] Support **multiple calls** or a deliberate single-call limit
- [ ] Handle **tool & external API errors**
- [ ] Apply **authentication & authorization** where required
- [ ] Require **confirmation for consequential actions**
- [ ] Add **unit tests** for tool functions
- [ ] Record **latency, outcomes, usage**
- [ ] Document **security assumptions & limitations**

### Suggested structure

```text
11-Tool-Calling-and-External-APIs/
├── README.md
├── requirements.txt
├── .env.example
├── app.py
├── src/
│   ├── config.py
│   ├── tool_schemas.py
│   ├── tool_registry.py
│   ├── tool_executor.py
│   ├── api_clients.py
│   ├── authorization.py
│   └── llm_client.py
├── tests/
│   ├── test_tool_schemas.py
│   ├── test_tool_executor.py
│   ├── test_authorization.py
│   └── test_api_clients.py
└── docs/
    └── security.md
```

### Stretch goals

1. FastAPI endpoints
2. DB-backed tool
3. Bounded retries
4. Idempotency keys for write ops
5. Approval step for sensitive actions
6. Structured logging & tracing
7. Simple web interface
8. Compare **direct SDK vs LangChain** tool calling
9. Evaluation tests for tool selection & argument correctness

---
## 🎤 12. Interview Questions

**Q1. What is tool calling?**  
Letting a model **request** an application execute a defined function using structured arguments.

**Q2. Does the LLM execute the function itself?**  
**No.** The app receives the request, validates it, executes if permitted, and returns the result.

**Q3. What is a tool schema?**  
A description of a tool's name, purpose, parameters, and expected argument structure.

**Q4. Is schema validation enough for security?**  
No. Auth, authz, business rules, and operational safeguards are separate requirements.

**Q5. Tool call vs normal response?**  
Normal response → content for the user. Tool call → a request for an application operation, handled by the protocol.

**Q6. What happens after a tool executes?**  
The app sends the tool result back using the matching `tool_call_id`. The model can then answer or request another permitted tool.

**Q7. How do you prevent arbitrary function execution?**  
Use an **explicit tool registry / allowlist**, validate names & args, never `eval` model-generated code.

**Q8. Risk of allowing arbitrary URLs?**  
**SSRF** — the model can reach internal services. Fix endpoints, validate params, restrict outbound network.

**Q9. How do you prevent duplicate write ops?**  
**Idempotency keys**, durable operation state, transactional semantics, and explicit retry rules.

**Q10. Is every tool-calling system an agent?**  
**No.** A simple app performs a single controlled tool call. Agentic systems add planning, state, feedback, or decision-making.

---
## ✅ 13. Module Completion Checklist

- [ ] Understand the tool-calling lifecycle
- [ ] Explain tool calling vs structured outputs vs RAG
- [ ] Define tools and argument schemas
- [ ] Implement a tool with Python
- [ ] Connect a model using a tool-calling API
- [ ] Execute calls through an **explicit allowlist**
- [ ] Return tool results to the model
- [ ] Understand controlled multi-step tool loops
- [ ] Integrate a trusted external API
- [ ] Validate inputs and enforce permissions
- [ ] Handle timeouts, failures, duplicate operations
- [ ] Test tools independently of the LLM
- [ ] Complete the **API-Connected AI Assistant**
- [ ] Document security & reliability limitations

---
## 🏁 14. Key Takeaways

- **Tool calling** connects model reasoning with **application capabilities**
- The model **requests** an operation; the **application decides** whether to execute it
- Tool **descriptions & schemas** help the model produce structured requests
- **Server-side validation and authorization** remain mandatory
- **External API integration** requires timeouts, error handling, and safe credential management
- **Side-effecting operations** need stronger safeguards than read-only ones
- Tool calling is a **building block for agentic systems** — but a simple tool loop is **not automatically a complete agent**

### 🧭 Golden Rule

> **The model proposes; the application disposes. Always validate, authorize, and control what actually executes.**

---

### 🔗 What's Next?

**Module 12 — LLM Evaluation and Guardrails**

- Evaluate LLM outputs systematically
- Add **guardrails** for input validation, output quality, policy compliance
- Build on the tool-calling assistant
- Test whether it works **and** whether it behaves reliably under **realistic & adversarial** inputs